In [6]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGA</cre>ATTACGG<bc/>AGATCGGA')\
                  .named('template_pool')

mutated_pool = template_pool.stylize(region='cre', style='goldenrod')\
                            .mutagenize(region='cre',
                                        mutation_rate=0.1, 
                                        style='yellow bold',
                                        mode='random', 
                                        num_states=10, 
                                        prefix='mut').named('mutated_pool')\

deletion_pool = template_pool.stylize(region='cre', style='salmon')\
                             .deletion_scan(region='cre', 
                                            deletion_length=6, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            style='red bold',
                                            prefix='del').named('deletion_pool')\
                            .repeat_states(2, prefix='v', iter_order=-2)

sites_pool=pp.from_seqs(['AAAAAA','TTTTTT'], 
                        mode='sequential', 
                        iter_order=-1,
                        prefix='site').named('sites_pool')

insertion_pool = template_pool.stylize(region='cre', style='blue')\
                              .insertion_scan(region='cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential',
                                              prefix='insscan',
                                              prefix_position='pos', 
                                              prefix_insert='ins',
                                              style='cyan bold').named('insertion_pool')
                              
shuffle_pool = template_pool.stylize(region='cre', style='purple')\
                            .shuffle_scan(region='cre', 
                                          shuffle_length=6, 
                                          shuffles_per_position=2,
                                          positions=slice(None, None, 5), 
                                          mode='sequential', 
                                          style='magenta bold',
                                          prefix='shufscan',
                                          prefix_position='pos',
                                          prefix_shuffle='shuf')

combo_pool = pp.stack([mutated_pool, deletion_pool, insertion_pool, shuffle_pool])\
    .named('stack_pool')\
    .insert_kmers(region='bc', mode='random', length=5, prefix='bc', style='green bold')\
    .named('combo_pool')\
    .stylize(which='tags', style='gray')

combo_pool.print_library(show_name=True,seed=12)

pool[15]: seq_length=None, num_states=40
state  name                           seq
    0  mut_0.bc_0                     TCCCGACT<cre>GAGAAGCGGGCAGTGAGCACACAGGA</cre>ATTACGG<bc>AGGCC</bc>AGATCGGA
    1  mut_1.bc_1                     TCCCGACT<cre>GGAAAGCGGGTAGTGAGCACACATGA</cre>ATTACGG<bc>TGCGG</bc>AGATCGGA
    2  mut_2.bc_2                     TCCCGACT<cre>GGAAAGCGGTCAGTAAGCACATAGGA</cre>ATTACGG<bc>TTCGG</bc>AGATCGGA
    3  mut_3.bc_3                     TCCCGACT<cre>GGAAAGCGGGCAGTTAGCATACATGA</cre>ATTACGG<bc>TTGCA</bc>AGATCGGA
    4  mut_4.bc_4                     TCCCGACT<cre>GGAACGCGGGCAGTGAGCACACAGGA</cre>ATTACGG<bc>CGCGG</bc>AGATCGGA
    5  mut_5.bc_5                     TCCCGACT<cre>GGAAAGCGGGCATTTAGCACACAGGA</cre>ATTACGG<bc>GCGCA</bc>AGATCGGA
    6  mut_6.bc_6                     TCCCGACT<cre>GGATAGCGGGCAGTCAGCACTCAGTA</cre>ATTACGG<bc>ATATG</bc>AGATCGGA
    7  mut_7.bc_7                     TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGG</cre>ATTACGG<bc>TGGAG</bc>AGATCGGA
    8  mut_8.

Pool(id=15, name='pool[15]', op='op[15]:stylize', num_states=40)

In [2]:
combo_pool.print_dag()

pool[14] (pool, n=30)
└── op[14]:stylize [mode=fixed, n=1]
    └── combo_pool (pool, n=30)
        └── op[13]:get_kmers [mode=random, n=30]
            └── stack_pool (pool, n=30)
                └── op[12]:stack [mode=sequential, n=4]
                    ├── mutated_pool (pool, n=5)
                    │   └── op[2]:mutagenize [mode=random, n=5]
                    │       └── pool[1] (pool, n=1)
                    │           └── op[1]:stylize [mode=fixed, n=1]
                    │               └── template_pool (pool, n=1)
                    │                   └── op[0]:from_seq [mode=fixed, n=1]
                    ├── deletion_pool (pool, n=5)
                    │   └── op[4]:deletion_scan [mode=sequential, n=5]
                    │       └── pool[3] (pool, n=1)
                    │           └── op[3]:stylize [mode=fixed, n=1]
                    │               └── template_pool (pool, n=1)
                    │                   └── op[0]:from_seq [mode=fixed, n=1]
    

Pool(id=14, name='pool[14]', op='op[14]:stylize', num_states=30)

In [3]:
combo_pool.mutagenize?

Signature:
combo_pool.mutagenize(
    region: Union[str, collections.abc.Sequence[numbers.Integral], NoneType] = None,
    num_mutations: Optional[numbers.Integral] = None,
    mutation_rate: Optional[numbers.Real] = None,
    allowed_chars: Optional[str] = None,
    style: Optional[str] = None,
    prefix: Optional[str] = None,
    mode: Literal['random', 'sequential', 'fixed'] = 'random',
    num_states: Optional[int] = None,
    iter_order: Optional[numbers.Real] = None,
) -> poolparty.pool.Pool
Docstring:
Create a Pool that applies mutations to a sequence.

Parameters
----------
region : Union[str, Sequence[Integral], None], default=None
    Region to mutagenize. Can be a marker name (str), explicit interval [start, stop],
    or None to mutagenize entire sequence. Positions are region-relative.
num_mutations : Optional[Integral], default=None
    Fixed number of mutations to apply (mutually exclusive with mutation_rate).
mutation_rate : Optional[Real], default=None
    Probability